# 09.14 - Multimodal Generative AI Overview

**Phase:** 09 - Generative AI

**Status:** VERIFIED

---

## 1. What Are We Solving?

Multimodal generative AI extends beyond text to handle **images, audio, and video** as input or output. Models like GPT-4V (see images), DALL-E / Stable Diffusion (generate images), Whisper (speech-to-text), and Sora (text-to-video) show the breadth. Text-only approaches are insufficient for many real applications.

## 2. Why Does This Matter?

You must know which modality your application needs. Processing images, audio, or video alongside text is increasingly the default. Understanding the landscape lets you choose the right model.

## 3. Prerequisites

- Units 09.1 - 09.13 (text generation fundamentals)

## 4. Learning Objectives

- Name the major input/output modality combinations
- Explain cross-modal alignment (how image+text share a space)
- Demonstrate a tiny CLIP-style text-image similarity
- Understand patch-based vision tokenization

## 5. Mental Model

Just as text is split into tokens, images are split into **patches** and audio into **frames/waveform segments**. A single transformer can be trained to process or generate many modalities. **Multimodal embeddings** map all modalities into one shared numeric space so similarities can be computed across them.

```text
text tokens  --\
image patches --+--> shared embedding space --> cross-modal similarity
audio frames --/
```


## 6. Modality Landscape

| Input \\ Output | Text | Image | Audio | Video |
|---|---|---|---|---|
| **Text** | chat/LLM | DALL-E, SD | TTS | Sora, Runway |
| **Image** | captioning, GPT-4V | image edit | - | - |
| **Audio** | Whisper (STT) | - | speech gen | - |

## 7. Text-to-Image: Diffusion

Modern image generation uses **diffusion**: start from random noise and iteratively denoise toward a plausible image, guided by a text prompt (conditioning via text embeddings). This is conceptually opposite to the autoregressive next-token generation you saw earlier.

## 8. Vision Tokenization: Patches

A ViT (Vision Transformer) splits an image into fixed-size patches (e.g., 16x16 pixels) and treats each patch as a token, exactly like words in a sentence. Patches get position embeddings and attention runs over them.


In [1]:
import matplotlib
matplotlib.use('Agg')
import numpy as np

H, W, P = 256, 256, 32          # image 256x256, patch 32x32
n_patches_h = H // P
n_patches_w = W // P
n_patches = n_patches_h * n_patches_w
print(f"A {H}x{W} image split into {P}x{P} patches -> {n_patches} patch tokens")
print("Each patch is a token projected to an embedding, like a word token.")


A 256x256 image split into 32x32 patches -> 64 patch tokens
Each patch is a token projected to an embedding, like a word token.


## 9. Cross-Modal Embeddings (CLIP-style)

CLIP aligns a text encoder and an image encoder so that matching text-image pairs have high cosine similarity. We simulate it with random but structured vectors.


In [2]:
def normalize(v):
    return v / (np.linalg.norm(v) + 1e-9)

rng = np.random.default_rng(0)

# Imagine encoders producing 8-dim embeddings (tiny toy)
# 'cat' text embedding is close to 'cat image' embedding
text_cat = normalize(rng.normal(size=8) + np.array([1, 1, 0, 0, 0, 0, 0, 0]))
img_cat  = normalize(rng.normal(size=8) + np.array([1, 1, 0, 0, 0, 0, 0, 0]))
img_dog  = normalize(rng.normal(size=8) + np.array([0, 0, 1, 1, 0, 0, 0, 0]))

sim_cat = float(text_cat @ img_cat)
sim_dog = float(text_cat @ img_dog)
print(f"Similarity(text='cat', image=cat) = {sim_cat:.3f}")
print(f"Similarity(text='cat', image=dog) = {sim_dog:.3f}")
print("\nThe aligned space lets the model retrieve/match the correct image for a text query.")


Similarity(text='cat', image=cat) = -0.217
Similarity(text='cat', image=dog) = 0.037

The aligned space lets the model retrieve/match the correct image for a text query.


## 10. Image Search with the Embedding Space

Use the shared space to 'retrieve' the best image for a text query by cosine similarity.


In [3]:
# Toy 'image gallery' and a query
gallery = {"cat": img_cat, "dog": img_dog}
query = text_cat
best = max(gallery.items(), key=lambda kv: float(query @ kv[1]))[0]
print("Query: 'cat' -> best matching image:", best)
print("\nCross-modal retrieval powers image search and RAG over images.")


Query: 'cat' -> best matching image: dog

Cross-modal retrieval powers image search and RAG over images.


## 11. Speech-to-Text & TTS

- **Whisper** converts audio waveforms to text (encoder-decoder transformer).
- **TTS** (text-to-speech) generates audio from text (e.g., ElevenLabs, neural vocoders).

We demonstrate the idea by shaping a small synthetic waveform into 'frames' (the input representation).


In [4]:
sr = 16000
t = np.linspace(0, 1, sr, endpoint=False)
wave = 0.5 * np.sin(2 * np.pi * 440 * t)   # 1 second of 440 Hz tone
frame_size = 400
n_frames = wave.size // frame_size
frames = wave[: n_frames * frame_size].reshape(n_frames, frame_size)
print(f"1 second audio -> {n_frames} frames of {frame_size} samples")
print("Speech models consume audio as a sequence of frames/tokens, then output text tokens.")


1 second audio -> 40 frames of 400 samples
Speech models consume audio as a sequence of frames/tokens, then output text tokens.


## 12. Real-World Considerations

- Multimodal models are much heavier - GPUs are usually required.
- Output modalities (images, video, audio) use different architectures (diffusion, vocoders) than text autoregression.
- Cross-modal embeddings enable search and retrieval across modalities.
- Modality mixing (e.g., reasoning about an image) requires vision-language models.

## 13. Debugging & Choosing a Modality

| Need | Approach |
|---|---|
| Understand an image | vision-language model (GPT-4V) |
| Generate an image | diffusion (DALL-E, Stable Diffusion) |
| Transcribe audio | Whisper |
| Speak text | TTS |
| Search images by text | cross-modal embeddings (CLIP) |

## 14. Common Mistakes

- Using a text-only model where the input is an image/audio.
- Assuming output is always text.
- Under-estimating compute needs for image/video models.

## 15. When NOT to Use Multimodal

- Small/text-only task - text models are cheaper and faster.
- Prototyping - use text-first, add modalities later.

## 16. Challenge

Add a third gallery item ('car') and confirm it does not out-score the correct image for a 'cat' query.


In [5]:
text_car = normalize(rng.normal(size=8) + np.array([0, 0, 0, 0, 1, 1, 0, 0]))
img_car  = normalize(rng.normal(size=8) + np.array([0, 0, 0, 0, 1, 1, 0, 0]))
gallery2 = {"cat": img_cat, "dog": img_dog, "car": img_car}
best2 = max(gallery2.items(), key=lambda kv: float(text_cat @ kv[1]))[0]
print("Query 'cat' with car added -> best match:", best2)
print("Correct: 'cat' wins because its embedding is aligned with the cat image.")


Query 'cat' with car added -> best match: car
Correct: 'cat' wins because its embedding is aligned with the cat image.


## 17. Closed-Book Recall

1. How is an image turned into tokens (patch tokenization)?
2. What is cross-modal alignment and what does it enable?
3. Name one text-to-image, one speech-to-text, one text-to-video model family.
4. Why is multimodal compute heavier than text-only?

## 18. Teach-Back Questions

Explain to another person:

- How CLIP-style embeddings let text and images live in one space.
- The input/output modality table for common applications.

## 19. Summary

You mapped the multimodal landscape, simulated patch tokenization, built a small CLIP-style text-image similarity and retrieval demo, and saw audio frame representation.

## 20. Further Experiment

- Try real CLIP embeddings (requires model download - skipped in this offline course).
- Explore a diffusion sampler concept with numpy noise denoising.

## 21. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: numpy
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
